<a href="https://colab.research.google.com/github/noname123920/diplom/blob/main/pinn_deepxde2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install deepxde
!pip install optuna

In [2]:
import os
os.environ["DDE_BACKEND"] = "pytorch"
import deepxde as dde
import numpy as np
import torch
import optuna

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


In [3]:
kappa = 1.0
geom = dde.geometry.Interval(0, np.pi)
timedomain = dde.geometry.TimeDomain(0, 1)
geomtime = dde.geometry.GeometryXTime(geom, timedomain)

def pde(x, p):
    dp_t = dde.grad.jacobian(p, x, i=0, j=1)
    dp_xx = dde.grad.hessian(p, x, i=0, j=0)
    return dp_t - kappa * dp_xx

def exact_solution(x):
    xx = x[:, 0:1]
    tt = x[:, 1:2]
    return np.exp(-tt) * np.sin(xx)

X_test = geomtime.uniform_points(1000)
y_true = exact_solution(X_test)

def objective(trial):
  dde.config.set_random_seed(42)
  iterations = trial.suggest_int("iterations", 1000, 5000, step=100)
  num_domain = trial.suggest_int("num_domain", 20, 500, step=10)

  data = dde.data.TimePDE(geomtime, pde, [], num_domain=num_domain, train_distribution="uniform")

  layer_size = [2] + [20] * 2 + [1]
  activation = "tanh"
  initializer = "Glorot uniform"
  net = dde.nn.FNN(layer_size, activation, initializer)

  def output_transform(x, y):
      x_col = x[:, 0:1]
      t_col = x[:, 1:2]
      return torch.sin(x_col) + x_col * (np.pi - x_col) * t_col * y

  net.apply_output_transform(output_transform)

  model = dde.Model(data, net)
  model.compile("adam", lr=1e-3)
  losshistory, train_state = model.train(iterations=iterations)

  y_pred = model.predict(X_test)
  l2_error = dde.metrics.l2_relative_error(y_true, y_pred)
  return float(l2_error)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

print(study.best_params)
print(study.best_value)

[I 2026-11-13 07:26:30,508] A new study created in memory with name: no-name-6230809f-ca62-48f0-a73b-17a88f212c81


Compiling model...
'compile' took 2.140926 s

Training model...

Step      Train loss    Test loss     Test metric
0         [1.34e+01]    [1.34e+01]    []  
1000      [1.52e-03]    [1.52e-03]    []  
1800      [4.40e-04]    [4.40e-04]    []  

Best model at step 1800:
  train loss: 4.40e-04
  test loss: 4.40e-04
  test metric: []

'train' took 13.247110 s



[I 2026-11-13 07:26:45,936] Trial 0 finished with value: 0.010100535117089748 and parameters: {'iterations': 1800, 'num_domain': 90}. Best is trial 0 with value: 0.010100535117089748.


Compiling model...
'compile' took 0.000198 s

Training model...

Step      Train loss    Test loss     Test metric
0         [1.31e+01]    [1.31e+01]    []  
1000      [1.53e-03]    [1.53e-03]    []  
1200      [1.08e-03]    [1.08e-03]    []  

Best model at step 1200:
  train loss: 1.08e-03
  test loss: 1.08e-03
  test metric: []

'train' took 5.101272 s



[I 2026-11-13 07:26:51,051] Trial 1 finished with value: 0.013375355862081051 and parameters: {'iterations': 1200, 'num_domain': 130}. Best is trial 0 with value: 0.010100535117089748.


Compiling model...
'compile' took 0.000276 s

Training model...

Step      Train loss    Test loss     Test metric
0         [1.33e+01]    [1.33e+01]    []  
1000      [1.51e-03]    [1.51e-03]    []  
2000      [3.57e-04]    [3.57e-04]    []  
3000      [1.48e-04]    [1.48e-04]    []  
4000      [6.49e-05]    [6.49e-05]    []  
4900      [4.07e-05]    [4.07e-05]    []  

Best model at step 4900:
  train loss: 4.07e-05
  test loss: 4.07e-05
  test metric: []

'train' took 27.378498 s



... (32 outputs left out)

[I 2026-11-13 07:31:58,673] Trial 18 finished with value: 0.013970027677714825 and parameters: {'iterations': 1000, 'num_domain': 340}. Best is trial 14 with value: 0.0013469541445374489.


Compiling model...
'compile' took 0.000139 s

Training model...

Step      Train loss    Test loss     Test metric
0         [1.22e+01]    [1.22e+01]    []  
1000      [1.57e-03]    [1.57e-03]    []  
2000      [3.75e-04]    [3.75e-04]    []  
3000      [1.48e-04]    [1.48e-04]    []  
4000      [6.66e-05]    [6.66e-05]    []  
5000      [4.25e-05]    [4.25e-05]    []  

Best model at step 5000:
  train loss: 4.25e-05
  test loss: 4.25e-05
  test metric: []

'train' took 26.943870 s



[I 2026-11-13 07:32:25,633] Trial 19 finished with value: 0.0013490355340763927 and parameters: {'iterations': 5000, 'num_domain': 500}. Best is trial 14 with value: 0.0013469541445374489.


{'iterations': 5000, 'num_domain': 470}
0.0013469541445374489


In [4]:
fig = optuna.visualization.plot_optimization_history(study)
fig.show()

In [5]:
fig = optuna.visualization.plot_slice(study)
fig.show()